# Notebook 4 — RAGAS Evaluation

This notebook measures the quality of the RAG pipeline using the RAGAS
evaluation framework. We build a test set of 10 questions grounded in the
loaded 3GPP specifications and score the pipeline on four metrics: faithfulness,
answer relevancy, context precision, and context recall. These scores give an
objective measure of retrieval and generation quality beyond just eyeballing
the answers.

In [1]:
import sys
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "3gpp-rag"

print(sys.executable)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))

c:\Users\nabee\anaconda3\envs\rag3gpp\python.exe
OpenAI key loaded: True


## Setting up the RAG chain and retriever

We rebuild the RAG chain and retriever from the existing ChromaDB so the
evaluation runs against the same pipeline used in production. We also keep
a reference to the retriever separately because RAGAS needs to inspect the
retrieved chunks directly, not just the final answer.

In [2]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db",
    embedding_function=embeddings
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a technical assistant specializing in 3GPP wireless standards.
Answer the question using ONLY the context provided below.
For every piece of information you use, cite the source document and page number.
If the answer is not in the context, say "I don't have enough information in the
loaded specifications to answer this question."

Context:
{context}

Question:
{question}

Answer (with citations):
""")

def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("title", "Unknown")
        page = doc.metadata.get("page", "Unknown")
        formatted.append(f"[{source}, Page {page}]\n{doc.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain and retriever ready")

RAG chain and retriever ready


## Test set

Ten questions drawn directly from the loaded specifications, covering all
eight documents in the corpus. Each question has a ground truth answer so
RAGAS can measure how well the pipeline performs against a known baseline.
These were written to reflect realistic queries an engineer or researcher
would ask when navigating 3GPP standards.

In [3]:
# ten question/ground-truth pairs covering all 8 specs in the corpus
test_set = [
    {
        "question": "What are the supported modulation orders Qm for PDSCH including QPSK 16QAM 64QAM 256QAM?",
        "ground_truth": "PDSCH supports QPSK with Qm=2, 16QAM with Qm=4, 64QAM with Qm=6, and 256QAM with Qm=8."
    },
    {
        "question": "What subcarrier spacing values are defined for NR numerology mu=0 and mu=1?",
        "ground_truth": "For numerology mu=0 the subcarrier spacing is 15 kHz, and for mu=1 it is 30 kHz."
    },
    {
        "question": "What is the maximum number of MIMO layers supported for PDSCH in NR?",
        "ground_truth": "NR supports up to 8 layers for PDSCH transmission."
    },
    {
        "question": "What are the two cyclic prefix types defined in NR?",
        "ground_truth": "NR defines normal cyclic prefix and extended cyclic prefix."
    },
    {
        "question": "What is the slot duration in milliseconds for NR numerology mu=1 with 30 kHz subcarrier spacing?",
        "ground_truth": "For numerology mu=1 with 30 kHz subcarrier spacing the slot duration is 0.5 milliseconds."
    },
    {
        "question": "How many resource elements are in one resource block in NR?",
        "ground_truth": "One NR resource block contains 12 subcarriers in the frequency domain."
    },
    {
        "question": "What are the frequency ranges FR1 and FR2 defined in NR?",
        "ground_truth": "FR1 covers 410 MHz to 7125 MHz and FR2 covers 24250 MHz to 52600 MHz."
    },
    {
        "question": "What is the purpose of the physical random access channel PRACH in NR?",
        "ground_truth": "PRACH is used by UEs to initiate random access and establish uplink synchronization with the gNB."
    },
    {
        "question": "What channel coding scheme is used for NR data channels?",
        "ground_truth": "NR uses low density parity check LDPC codes for data channel coding."
    },
    {
        "question": "What is the role of the NG-RAN architecture in 5G NR?",
        "ground_truth": "NG-RAN provides the radio access network functions in 5G connecting UEs to the 5G core network through gNBs."
    },
]

print(f"Test set created: {len(test_set)} questions")
for i, item in enumerate(test_set):
    print(f"  Q{i+1}: {item['question'][:70]}...")

Test set created: 10 questions
  Q1: What are the supported modulation orders Qm for PDSCH including QPSK 1...
  Q2: What subcarrier spacing values are defined for NR numerology mu=0 and ...
  Q3: What is the maximum number of MIMO layers supported for PDSCH in NR?...
  Q4: What are the two cyclic prefix types defined in NR?...
  Q5: What is the slot duration in milliseconds for NR numerology mu=1 with ...
  Q6: How many resource elements are in one resource block in NR?...
  Q7: What are the frequency ranges FR1 and FR2 defined in NR?...
  Q8: What is the purpose of the physical random access channel PRACH in NR?...
  Q9: What channel coding scheme is used for NR data channels?...
  Q10: What is the role of the NG-RAN architecture in 5G NR?...


## Running the pipeline on the test set

We run each question through the full RAG chain and also collect the retrieved
chunks separately. RAGAS needs both the generated answer and the raw retrieved
contexts to compute its metrics — the answer alone is not enough.

In [4]:
from langchain_core.documents import Document

results = []

for i, item in enumerate(test_set):
    question = item["question"]
    ground_truth = item["ground_truth"]

    # get retrieved chunks for this question
    retrieved_docs = retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]

    # get the generated answer
    answer = rag_chain.invoke(question)

    results.append({
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": ground_truth
    })

    print(f"Q{i+1} done: {question[:60]}...")

print(f"\nAll {len(results)} questions processed")

Q1 done: What are the supported modulation orders Qm for PDSCH includ...
Q2 done: What subcarrier spacing values are defined for NR numerology...
Q3 done: What is the maximum number of MIMO layers supported for PDSC...
Q4 done: What are the two cyclic prefix types defined in NR?...
Q5 done: What is the slot duration in milliseconds for NR numerology ...
Q6 done: How many resource elements are in one resource block in NR?...
Q7 done: What are the frequency ranges FR1 and FR2 defined in NR?...
Q8 done: What is the purpose of the physical random access channel PR...
Q9 done: What channel coding scheme is used for NR data channels?...
Q10 done: What is the role of the NG-RAN architecture in 5G NR?...

All 10 questions processed


## Computing RAGAS metrics

We pass the results to RAGAS which uses GPT-4o-mini as a judge to score
each question on four metrics. This takes 2-3 minutes and makes additional
OpenAI API calls. The final scores are averaged across all 10 questions.

In [5]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
import numpy as np

# wrap the models for RAGAS
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# convert results to a HuggingFace Dataset format that RAGAS expects
dataset = Dataset.from_list(results)

# run evaluation - takes 2-3 minutes
print("Running RAGAS evaluation...")
scores = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)


print("\nRAGAS Evaluation Results:")
print(f"  Faithfulness:      {np.mean(scores['faithfulness']):.3f}")
print(f"  Answer relevancy:  {np.mean(scores['answer_relevancy']):.3f}")
print(f"  Context precision: {np.mean(scores['context_precision']):.3f}")
print(f"  Context recall:    {np.mean(scores['context_recall']):.3f}")

c:\Users\nabee\anaconda3\envs\rag3gpp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\nabee\AppData\Local\Temp\ipykernel_8740\3075083494.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\nabee\AppData\Local\Temp\ipykernel_8740\3075083494.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precisi

Running RAGAS evaluation...


Evaluating: 100%|██████████| 40/40 [00:42<00:00,  1.05s/it]



RAGAS Evaluation Results:
  Faithfulness:      0.675
  Answer relevancy:  0.628
  Context precision: 0.675
  Context recall:    0.750
